### Goal of notebook: process ambimux output
- Summarize # barcodes outside of cellranger calling method
- Check ambient RNA/DNA distributions per sample

In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
samples = list(np.loadtxt(f'{projdir}/txt/samples.txt', dtype=str))
times = list(np.loadtxt(f'{projdir}/txt/times.txt', dtype=str))
clusters = 4

In [ ]:
cr_base = f'{projdir}/mapping/cr_arc/igvf_ref/default/'

In [ ]:
s = samples[0]
barcodes = pd.read_csv(f'{cr_base}/{s}/outs/filtered_feature_bc_matrix/barcodes.tsv.gz',
                       sep='\t', header=None, index_col=0)
barcodes['sample'] = s

for s in samples:
    tmp_bcs = pd.read_csv(f'{cr_base}/{s}/outs/filtered_feature_bc_matrix/barcodes.tsv.gz',
                       sep='\t', header=None, index_col=0)
    tmp_bcs['sample'] = s
    barcodes = pd.concat([barcodes, tmp_bcs])

barcodes.index = [f'{x}_{y}' for x,y in zip(barcodes.index,barcodes['sample'])]
barcodes = barcodes[~barcodes.index.duplicated()]
barcodes.shape

In [ ]:
indir = f'{projdir}/demux/wgs/ambimux/joint/'

s = samples[0]
ambimux = pd.read_csv(f'{indir}/{s}.summary.txt', sep='\t', index_col=0)
ambimux['sample'] = s

for s in samples[1:]:
    tmp = pd.read_csv(f'{indir}/{s}.summary.txt', sep='\t', index_col=0)
    tmp['sample'] = s
    ambimux = pd.concat([ambimux, tmp], axis=0)

ambimux['barcode'] = ambimux.index
ambimux.index = [f'{x}_{y}' for x,y in zip(ambimux['barcode'],ambimux['sample'])]

d_mask = ambimux['best_type'] == 'Doublet'
u_mask = ambimux['best_type'] == 'Empty'

ambimux['assignment'] = ambimux['best_singlet'].copy()
ambimux.loc[d_mask,'assignment'] = 'doublet'
ambimux.loc[u_mask,'assignment'] = 'unassigned'

ambimux['droplet_type'] = ambimux['assignment'].copy()
for donor in donors:
    ambimux['droplet_type'].replace({donor : 'singlet'}, inplace=True)
    
ambimux.shape

In [ ]:
ambimux['sample'].value_counts().sort_index(), \
ambimux['assignment'].value_counts().sort_index(), \
ambimux['droplet_type'].value_counts().sort_index()

In [ ]:
mask = [x in donors for x in ambimux['assignment']]
ambimux_sings = ambimux[mask]
ambimux_sings.shape

In [ ]:
ambimux_sings['sample'].value_counts().sort_index(), \
ambimux_sings['assignment'].value_counts().sort_index()

In [ ]:
ambimux_sings.head()

In [ ]:
bc_mask = barcodes.index.isin(ambimux_sings.index)
tmp_barcodes = barcodes[bc_mask]
tmp_barcodes.shape

In [ ]:
ambimux_sings['cellranger_passqc'] = False
ambimux_sings.loc[tmp_barcodes.index,'cellranger_passqc'] = True

In [ ]:
### cause for concern? less barcodes than using cellranger default reference
### comparison between references is done in another notebook
ambimux_sings['cellranger_passqc'].value_counts()

In [ ]:
demux_time = pd.read_csv(f'{projdir}/txt/demux_time_map.txt', sep='\t', header=0, index_col=None)
demux_map = dict(zip(demux_time['Sample'], demux_time['TimeSample2']))

ambimux_sings['sample_donor'] = [f'{x}_{y}' for x,y in zip(ambimux_sings['sample'], ambimux_sings['assignment'])]
ambimux_sings['time'] = [demux_map[x].split('_')[1] for x in ambimux_sings['sample_donor']]
ambimux_sings['time_num'] = [int(x[1:]) for x in ambimux_sings['time']]
ambimux_sings['donor_time'] = [f'{x}_{y}' for x,y in zip(ambimux_sings['assignment'],ambimux_sings['time_num'])]

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 8))
plt.subplots_adjust(wspace=0.2, hspace=0.5)

for i,ax in enumerate(axes.flatten()):
    s = samples[i]
    tmp_ambimux = ambimux[ambimux['sample'] == s]
    sns.kdeplot(tmp_ambimux['best_rna_ambient'], ax=ax)
    ax.set_xlabel('Ambient RNA %')
    ax.set_ylabel('')
    ax.set_title(s)
    ax.set_yticklabels([])
 
plt.suptitle('Ambimux (joint) inferred ambient RNA, all droplets', fontsize=20)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 8))
plt.subplots_adjust(wspace=0.2, hspace=0.5)

for i,ax in enumerate(axes.flatten()):
    s = samples[i]
    tmp_ambimux = ambimux[ambimux['sample'] == s]
    sns.kdeplot(tmp_ambimux['best_atac_ambient'], ax=ax)
    ax.set_xlabel('Ambient RNA %')
    ax.set_ylabel('')
    ax.set_title(s)
    ax.set_yticklabels([])
    
plt.suptitle('Ambimux (joint) inferred ambient DNA, all droplets', fontsize=20)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 8))
plt.subplots_adjust(wspace=0.2, hspace=0.5)

for i,ax in enumerate(axes.flatten()):
    s = samples[i]
    tmp_ambimux = ambimux[ambimux['sample'] == s]
    mask = [x in donors for x in tmp_ambimux['assignment']]
    tmp_ambimux = tmp_ambimux[mask]
    sns.kdeplot(tmp_ambimux['best_rna_ambient'], ax=ax)
    ax.set_xlabel('Ambient RNA %')
    ax.set_ylabel('')
    ax.set_title(s)
    ax.set_yticklabels([])
    
plt.suptitle('Ambimux (joint) inferred ambient RNA, singlets', fontsize=20)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 8))
plt.subplots_adjust(wspace=0.2, hspace=0.5)

for i,ax in enumerate(axes.flatten()):
    s = samples[i]
    tmp_ambimux = ambimux[ambimux['sample'] == s]
    mask = [x in donors for x in tmp_ambimux['assignment']]
    tmp_ambimux = tmp_ambimux[mask]
    sns.kdeplot(tmp_ambimux['best_atac_ambient'], ax=ax)
    ax.set_xlabel('Ambient DNA %')
    ax.set_ylabel('')
    ax.set_title(s)
    ax.set_yticklabels([])
    
plt.suptitle('Ambimux (joint) inferred ambient DNA, singlets', fontsize=20)

In [ ]:
n_singlets = ambimux_sings.shape[0]
amb_rna = np.mean(ambimux_sings['best_rna_ambient'])
amb_dna = np.mean(ambimux_sings['best_atac_ambient'])

print(f'Ambimux singlets : {n_singlets}')
print(f'Ambimux mean ambient RNA : {amb_rna}')
print(f'Ambimux mean ambient DNA : {amb_dna}')

### breakdown by sample

In [ ]:
singlet_summary = pd.DataFrame(columns=['# singlets'], index=samples)

for s in samples:
    tmp_ambimux = ambimux[ambimux['sample'] == s]
    singlet_summary.loc[s] = tmp_ambimux['droplet_type'].value_counts()['singlet']
    
singlet_summary

In [ ]:
ambient_summary = pd.DataFrame(columns=['Ambient RNA', 'Ambient DNA %'], index=samples)

for s in samples:
    tmp_ambimux = ambimux[ambimux['sample'] == s]
    amb_rna = np.mean(tmp_ambimux['best_rna_ambient'])
    amb_dna = np.mean(tmp_ambimux['best_atac_ambient'])
    ambient_summary.loc[s] = [amb_rna, amb_dna]
ambient_summary

In [ ]:
pd.DataFrame(ambimux_sings['sample'].value_counts().sort_index())

In [ ]:
ambimux.to_csv(f'{projdir}/csv/demux/ambimux/wgs/ambimux_joint.csv', sep='\t', header=True, index=True)
ambimux_sings.to_csv(f'{projdir}/csv/demux/ambimux/wgs/ambimux_joint_sings.csv', sep='\t', header=True, index=True)

In [ ]:
outdir = f'{projdir}/txt/barcodes/ambimux/wgs/sample/'
for s in samples:
    print(s)
    tmp_ambimux = ambimux_sings[ambimux_sings['sample'] == s].sort_values(by='barcode')[['barcode']]
    tmp_ambimux.to_csv(f'{outdir}/{s}.txt', sep='\t', header=None, index=None)

In [ ]:
donor_time = ambimux_sings['donor_time'].unique()
outdir = f'{projdir}/txt/barcodes/ambimux/wgs/donor_time/'
for s in donor_time:
    print(s)
    tmp_ambimux = ambimux_sings[ambimux_sings['donor_time'] == s].sort_values(by='barcode')[['barcode']]
    tmp_ambimux.to_csv(f'{outdir}/{s}.txt', sep='\t', header=None, index=None)